In [1]:
import time
from IPython.display import display, HTML, clear_output

class AutomatoDePilhaVisualDashboard:
    def __init__(self):
        self.transicoes = {}
        self.pilha = []
        self.configurar_transicoes()

    def configurar_transicoes(self):
        estados_ativos = [
            'q_inicio', 'q_lendo_paren', 'q_lendo_colch', 'q_lendo_chave',
            'q_fechando_paren', 'q_fechando_colch', 'q_fechando_chave'
        ]
        simbolos_topo = ['$', '(', '[', '{']

        for estado in estados_ativos:
            for topo in simbolos_topo:
                self.transicoes[(estado, '(', topo)] = ('q_lendo_paren', [topo, '('])
                self.transicoes[(estado, '[', topo)] = ('q_lendo_colch', [topo, '['])
                self.transicoes[(estado, '{', topo)] = ('q_lendo_chave', [topo, '{'])

            self.transicoes[(estado, ')', '(')] = ('q_fechando_paren', [])
            self.transicoes[(estado, ']', '[')] = ('q_fechando_colch', [])
            self.transicoes[(estado, '}', '{')] = ('q_fechando_chave', [])
            self.transicoes[(estado, '#', '$')] = ('q_aceita', ['$'])

    def obter_simbolos_esperados(self, estado_atual):
        if not self.pilha:
            return []
        topo_atual = self.pilha[-1]
        simbolos_validos = []
        for (estado, simbolo, topo) in self.transicoes.keys():
            if estado == estado_atual and topo == topo_atual:
                simbolos_validos.append(simbolo)
        return simbolos_validos

    def desenhar_estrutura(self, passo, estado, proximo_estado, entrada_completa, idx_atual, simbolos_esperados, aceito=None):
        cor_fundo = "#f8f9fa"
        if aceito is True:
            cor_fundo = "#d4edda"
            simbolos_esperados = []
        elif aceito is False:
            cor_fundo = "#f8d7da"
            simbolos_esperados = []
            
        fita_html = "<div style='margin: 10px 0; font-size: 1.4em; letter-spacing: 8px; text-align: center; font-family: monospace;'>"
        for idx, char in enumerate(entrada_completa):
            if idx == idx_atual and aceito is None:
                fita_html += f"<span style='background-color: #ffc107; color: #000; padding: 4px 10px; border-radius: 4px; font-weight: bold; border: 2px solid #ff9800; box-shadow: 0 2px 4px rgba(0,0,0,0.15);'>{char}</span>"
            else:
                fita_html += f"<span style='color: #adb5bd; padding: 4px;'>{char}</span>"
        fita_html += "</div>"

        if simbolos_esperados:
            formatados = [f"<code style='background: #e9ecef; border: 1px solid #ced4da; padding: 4px 8px; border-radius: 4px; color: #198754; font-weight: bold; font-size: 1.1em;'>{s}</code>" if s != '#' else "<code style='background: #e9ecef; border: 1px solid #ced4da; padding: 4px 8px; border-radius: 4px; color: #198754; font-weight: bold; font-size: 1.1em;'>#</code>" for s in simbolos_esperados]
            esperados_html = " ".join(formatados)
        else:
            esperados_html = "<span style='color: #dc3545; font-style: italic; font-weight: bold; font-size: 0.9em;'>Nenhum (Parada)</span>"

        badge_canto_esquerdo = f"""
        <div style="position: absolute; top: 20px; left: 20px; background: #ffffff; border: 2px solid #adb5bd; border-radius: 8px; padding: 12px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); width: 150px; text-align: center; z-index: 10;">
            <div style="font-size: 0.75em; color: #495057; font-weight: 800; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 1px solid #dee2e6; padding-bottom: 6px;">
                Símbolos Válidos<br><span style="color: #d63384;">({estado})</span>
            </div>
            <div style="display: flex; gap: 6px; flex-wrap: wrap; justify-content: center;">
                {esperados_html}
            </div>
        </div>
        """

        pilha_html = "<div style='display: flex; align-items: center; justify-content: center; gap: 6px; margin: 15px 0; padding: 5px; overflow-x: auto;'>"
        for idx, item in enumerate(self.pilha):
            is_topo = (idx == len(self.pilha) - 1)
            if item == '$':
                cor_item = "#ffeeba"
                rotulo = "<span style='display:block; font-size:0.55em; color:#7f6000; font-weight:normal; margin-top:2px;'>FUNDO</span>"
            elif is_topo and aceito is None:
                cor_item = "#bde0fe"
                rotulo = "<span style='display:block; font-size:0.55em; color:#003049; font-weight:bold; margin-top:2px;'>TOPO</span>"
            else:
                cor_item = "#e9ecef"
                rotulo = "<span style='display:block; font-size:0.55em; color:#6c757d; font-weight:normal; margin-top:2px;'>&nbsp;</span>"

            pilha_html += f"""
                <div style="border: 2px solid #495057; padding: 6px 12px; text-align: center; font-size: 1.3em; background-color: {cor_item}; font-weight: bold; border-radius: 6px; min-width: 45px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); line-height: 1.1;">
                    {item}
                    {rotulo}
                </div>
            """
            if idx < len(self.pilha) - 1:
                pilha_html += "<span style='color: #6c757d; font-weight: bold; font-size: 1.1em;'>→</span>"
        
        if not self.pilha:
            pilha_html += "<span style='color: #dc3545; font-style: italic;'>Pilha Totalmente Vazia!</span>"
        pilha_html += "</div>"
        
        if aceito is None:
            cor_destino = "#0dcaf0" if proximo_estado != 'q_rejeita' else "#dc3545"
            texto_destino = f"➔ {proximo_estado}"
        else:
            cor_destino = "#198754" if aceito else "#dc3545"
            texto_destino = "Parada Final"

        linha_destino = f"""
        <tr>
            <td style="padding: 4px 0; text-align: center;"><b>Transição Prevista (Destino):</b> <code style="font-size: 1.1em; background: #e9ecef; padding: 2px 8px; border-radius: 4px; color: {cor_destino}; font-weight: bold;">{texto_destino}</code></td>
        </tr>
        """
        
        html_final = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: {cor_fundo}; padding: 25px; border-radius: 12px; width: 720px; position: relative; border: 1px solid #dee2e6; box-shadow: 0 6px 12px rgba(0,0,0,0.1); margin: 0 auto;">
            
            {badge_canto_esquerdo}

            <h3 style="margin-top: 10px; margin-bottom: 20px; color: #212529; text-align: center; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;">Simulador Dinâmico de AP</h3>
            
            <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 1.05em;">
                <tr>
                    <td style="padding: 4px 0; text-align: center;"><b>Passo de Execução:</b> <span style="color: #0d6efd; font-weight:bold; font-size: 1.1em;">{passo:02d}</span></td>
                </tr>
                {linha_destino}
            </table>
            
            <h5 style="margin: 25px 0 5px 0; color: #495057; font-weight: 600; text-align: center;">Fita de Entrada (Cabeçote de Leitura):</h5>
            {fita_html}
            
            <h4 style="text-align: center; margin: 30px 0 5px 0; color: #495057; font-weight: 600; border-top: 1px dashed #ced4da; padding-top: 15px;">ESTRUTURA DA PILHA (Cresce p/ Direita)</h4>
            {pilha_html}
        </div>
        """
        
        clear_output(wait=True)
        display(HTML(html_final))
        time.sleep(2.0)

    def processar(self, string_entrada):
        self.pilha = ['$']
        estado_atual = 'q_inicio'
        entrada_completa = list(string_entrada) + ['#']
        passo = 0

        for idx, simbolo_lido in enumerate(entrada_completa):
            if estado_atual in ['q_aceita', 'q_rejeita']:
                break

            simbolos_esperados = self.obter_simbolos_esperados(estado_atual)
            
            topo_atual = self.pilha[-1]
            transicao_futura = self.transicoes.get((estado_atual, simbolo_lido, topo_atual))
            proximo_estado_previsto = transicao_futura[0] if transicao_futura else 'q_rejeita'

            self.desenhar_estrutura(passo, estado_atual, proximo_estado_previsto, entrada_completa, idx, simbolos_esperados)

            topo_pilha = self.pilha.pop()
            chave_transicao = (estado_atual, simbolo_lido, topo_pilha)

            if chave_transicao in self.transicoes:
                proximo_estado, simbolos_empilhar = self.transicoes[chave_transicao]
                for s in simbolos_empilhar:
                    self.pilha.append(s)
                estado_atual = proximo_estado
            else:
                estado_atual = 'q_rejeita'
                self.pilha.append(topo_pilha) 
                break
            
            passo += 1

        if estado_atual == 'q_aceita':
            self.desenhar_estrutura(passo, estado_atual, None, entrada_completa, len(entrada_completa)-1, [], aceito=True)
        else:
            self.desenhar_estrutura(passo, estado_atual, None, entrada_completa, idx, [], aceito=False)

In [2]:
exemplo_1 = AutomatoDePilhaVisualDashboard()

exemplo_1.processar("{[()]}")

Passo de Execução: 07
Transição Prevista (Destino): Parada Final


In [3]:
exemplo_2 = AutomatoDePilhaVisualDashboard()

exemplo_2.processar("{(})")

Passo de Execução: 02
Transição Prevista (Destino): Parada Final


In [4]:
exemplo_3 = AutomatoDePilhaVisualDashboard()

exemplo_3.processar("([]")

Passo de Execução: 03
Transição Prevista (Destino): Parada Final
